# RobustWorld: post-train V-JEPA 2 on Colab

Fine-tunes V-JEPA 2's **predictor** on your clips so its imagined future can learn the blockade,
then scores it against the pretrained model with the same frozen evaluation decoder
(see `CLAUDE.md`). The encoder stays frozen, so its features are computed once and cached.

1. Upload your clips (a zip from `python scripts/pack_clips.py` on your computer).
2. Encode every clip once (frozen encoder).
3. Score the **pretrained** predictor (the "before").
4. Post-train one predictor per cross-validation fold.
5. Score the **post-trained** predictors (the "after") and draw the blocker figures.

**Runtime:** GPU. An **A100** or **L4** is best (bf16). A **T4** works (fp16) but is slower.

Everything stays in this Colab session; the last cell downloads the results as a zip.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, subprocess
from google.colab import userdata

REPO = "/content/RobustWorld"
BRANCH = "posttrain-right-clips"   # switch to "main" once this is merged
if not os.path.exists(REPO):
    try:  # only needed if the repo is private again
        url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/FinPThomas/RobustWorld.git"
    except Exception:
        url = "https://github.com/FinPThomas/RobustWorld.git"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, REPO], check=True, capture_output=True)
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin",
                    "https://github.com/FinPThomas/RobustWorld.git"], check=True)
else:  # already cloned: update to the latest code (the token is only needed for a private repo)
    try:
        url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/FinPThomas/RobustWorld.git"
    except Exception:
        url = "https://github.com/FinPThomas/RobustWorld.git"
    r = subprocess.run(["git", "-C", REPO, "pull", "--ff-only", url, BRANCH], capture_output=True, text=True)
    print(r.stdout or r.stderr)
%cd {REPO}
!git log --oneline -1   # check this matches the newest commit on GitHub before training

In [ ]:
!pip install -q -e . -r models/vjepa2/requirements.txt

## Optional: keep outputs on Google Drive (recommended overnight)

Colab wipes `/content` when a runtime is reset. Run this **before** step 1 to keep the encoded
features, scores and grid progress in `MyDrive/RobustWorld/outputs`, so a restarted runtime can
carry on where it stopped. The feature cache takes several GB of Drive space. You can also put
`robustworld_clips.zip` in `MyDrive/` and step 1 will find it without uploading.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, shutil
keep = "/content/drive/MyDrive/RobustWorld/outputs"
os.makedirs(keep, exist_ok=True)
if not os.path.islink(f"{REPO}/outputs"):
    shutil.copytree(f"{REPO}/outputs", keep, dirs_exist_ok=True)   # keep anything already here
    shutil.rmtree(f"{REPO}/outputs")
    os.symlink(keep, f"{REPO}/outputs")
!ls -la {REPO}/outputs

## 1. Upload the clips

On your computer, after `robustworld` has cut the clips, run `python scripts/pack_clips.py` and
upload `outputs/robustworld_clips.zip` here. It holds the **right** clips only (ball rolled in from
the right; `left` is held out for testing later, and hand/excluded clips are not packed), their
manifests and the ball tracking. Every step below reads `manifest_right.jsonl`.

In [ ]:
import glob, zipfile
from google.colab import files
zips = (glob.glob("/content/robustworld_clips*.zip") + glob.glob(f"{REPO}/robustworld_clips*.zip")
        + glob.glob("/content/drive/MyDrive/robustworld_clips*.zip"))
if not zips:
    zips = list(files.upload())            # saved in the current folder (the repo)
zipfile.ZipFile(zips[0]).extractall(REPO)
!wc -l data/processed/clips/manifest_right.jsonl

## Overnight: run everything unattended

One cell does it all: encodes any clips not yet cached, then trains and scores every variant
before and after. After **each** run it rebuilds the summary and commits the results to
`results/<date>_<time>_overnight/` on GitHub (needs the `GITHUB_TOKEN` secret, see step 6), so a
disconnect keeps everything finished so far. A run that fails (e.g. out of memory) is logged in
`grid.json` and skipped; the rest carry on. `--resume` skips runs already done, so after a
disconnect just rerun the setup cells, then this cell.

**Colab limits to know about:**
- Sessions end after at most about 12 hours (free tier; longer on Pro), and idle tabs can be
  disconnected after about 90 minutes. Leave the tab open and the computer awake and plugged in.
  Background execution (runs with the tab closed) needs Colab Pro+.
- GPU time is rationed: if Colab refuses a GPU, try later or use a paid tier.
- If the runtime dies: reconnect, run the setup cells (clone, install, Drive, step 1), then this
  cell again. With outputs on Drive it resumes; without, it starts a new grid (results already on
  GitHub stay there).
- In the morning: open `results/README.md` on the branch on GitHub, or `grid.json` for what failed.

In [ ]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
import os
from google.colab import userdata
os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")    # needed to push results
EPOCHS = 10
VARIANTS = "plain codes rollout codes_rollout commit"
!python models/vjepa2/posttrain.py encode
!python models/vjepa2/experiments.py run --resume --save-as overnight --push --branch {BRANCH} \
    --note "overnight grid, {EPOCHS} epochs" --epochs {EPOCHS} --variants {VARIANTS} \
    --extra --batch-size 1 --accum 8

## 2. Encode every clip once

Runs the frozen encoder over each clip (full clip and context half alone) and caches the
features in `outputs/vjepa2/cache/`. Clips that are already cached are skipped.

In [ ]:
!python models/vjepa2/posttrain.py encode

### Optional: quick scale check

A 16-clip reproduction of the feature-scale mismatch and its fix (12 train, 4 held out, 3 epochs).
It prints token spreads, held-out L1 and the decoder's peak P(ball) on imagined frames, before and after.

In [ ]:
!python models/vjepa2/scale_check.py --clips 16 --epochs 3

## 3. Before: the pretrained predictor

Cross-validated ball readout and the blocker figures for the pretrained model.

In [ ]:
!python models/vjepa2/ball_probe_cv.py
!python models/vjepa2/blocker_figs.py --label pretrained --out outputs/vjepa2/blocker/pretrained

## 4. The experiment grid: before and after post-training

Four ways to run V-JEPA's predictor, each scored **before** (the pretrained predictor, 0 epochs)
and **after** post-training, with the same frozen decoder and folds:

| variant | what changes |
|---|---|
| `plain` | V-JEPA's own latent L1, all future steps at once (the baseline) |
| `codes` | predicts one of 256 discrete feature codes per token (cross-entropy), snapped at inference, so it can't output a blur |
| `rollout` | predicts one step at a time and feeds its own prediction back in |
| `codes_rollout` | both |
| `commit` | L1 weighted towards moving tokens, plus a contrast term that punishes a blurred average |

The codebook is k-means on each fold's own training features, so it learns nothing from outcomes.
Try `--variants plain --extra --fold 0 --epochs 2` first to check timing.

In [ ]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
EPOCHS = 10
VARIANTS = "plain codes rollout codes_rollout commit"
# batch 1 x 8 accumulation steps = the same effective batch of 8, in half the memory (fits a T4)
!python models/vjepa2/experiments.py run --epochs {EPOCHS} --variants {VARIANTS} --extra --batch-size 1 --accum 8

## 5. Results

Read the **threshold-free** columns first. They assume there is one ball and normalise each
imagined step's map, so a blurred but well-placed prediction still counts:
`one_ball_p_correct` (mean probability of the true outcome, each outcome weighted equally),
`one_ball_auroc` (through vs blocked), `argmax_hit_rate` (most likely cell within 48 px of the
real ball) and `cell_auroc`. The `*_0.5` columns use the old 0.5 cut-off and stay near their
floor until predictions are sharp. `enc_space_*` is the alternative reading (γ·pred+β into
encoder space, decoder without layer norm). The dashed line is the real future frames.

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open("outputs/vjepa2/experiments/summary.md").read()))
display(Image("outputs/vjepa2/experiments/summary.png"))

### Blocker figure for one run

P(blocked) against where the ball meets the plank, after (solid) over before (dashed).

In [ ]:
V = "codes"
for phase in ("before", "after"):
    extra = f"--before outputs/vjepa2/blocker/{V}-before/crossing.json" if phase == "after" else ""
    !python models/vjepa2/blocker_figs.py --per-clip outputs/vjepa2/ball_probe_cv/{V}-{phase}/per_clip.json \
        --label {V}-{phase} --out outputs/vjepa2/blocker/{V}-{phase} {extra}
display(Image(f"outputs/vjepa2/blocker/{V}-after/blocked_vs_crossing.png"))

## 6. Save the results to GitHub

Copies the scores, figures, summary table and training logs (not the feature cache or weights)
into `results/<date>_<time>_<NAME>/` with a README.md, rebuilds the index in `results/README.md`,
and commits and pushes them to `BRANCH`. Every save gets a new folder, so earlier runs are never
overwritten. Set `NAME` and `NOTE` to say what this run tried.

Pushing needs a Colab secret called `GITHUB_TOKEN` (key icon on the left): a GitHub fine-grained
token for FinPThomas/RobustWorld with **Contents: read and write**. Without it the results are
still saved in the Colab copy of the repo and you can download them below.

In [ ]:
NAME = "grid"                    # short label: letters, digits, - and _
NOTE = ""                        # one line on what this run tried
try:
    os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
    PUSH = "--push"
except Exception:
    print("No GITHUB_TOKEN secret: saving locally only")
    PUSH = ""
!python scripts/save_results.py --name "{NAME}" --note "{NOTE}" --branch {BRANCH} {PUSH}

## Download results (optional)

Zips everything in `outputs/vjepa2` (not the feature cache) and the training logs. Set
`WITH_CHECKPOINTS = True` to include the post-trained predictors too (about 90 MB per fold).

In [ ]:
import shutil, pathlib
WITH_CHECKPOINTS = False
out = pathlib.Path("/content/robustworld_posttrain")
shutil.rmtree(out, ignore_errors=True)
shutil.copytree("outputs/vjepa2", out / "outputs/vjepa2", ignore=shutil.ignore_patterns("cache"))
ck = pathlib.Path("checkpoints/vjepa2")
shutil.copytree(ck, out / "checkpoints/vjepa2", ignore=None if WITH_CHECKPOINTS else shutil.ignore_patterns("*.pt"))
files.download(shutil.make_archive(str(out), "zip", out))